[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C47_RecSys_Ranking_Course/03_two_tower_retrieval/03_two_tower_retrieval.ipynb)

# 03 · 双塔召回（用 numpy 从零）

目标：从零实现**双塔模型 + in-batch 负采样 + logQ 校正**——理解**分数矩阵 = UVᵀ**、**对角是正例的 softmax 损失**、**采样偏差与 logQ 校正**，训练后算 **Recall@K**。

路线：数据(隐式正例对) → 双塔前向 → in-batch softmax 损失(对拍) → 反向 + 训练看损失下降 → logQ 校正 → 点积召回 Recall@K → ✏️ 练习 → 📖 答案 → 🧪 真实 MovieLens 召回胶囊。

> 核心心智：**两塔独立编码 → 点积匹配 → 物品向量可离线建 ANN → 用 in-batch 负例 softmax 训练。**

## 0 · 数据：隐式正例对 (user, item)

复用模块 00 加载器，取高分(>=4)交互当**隐式正例对**。双塔训练只需正例对（负例靠 in-batch 采样）。

> 为让纯 numpy 训练秒级跑完，这里把数据**截断到前 300 用户 × 600 物品**（`cap_users/cap_items`）。机制完全一致，只是规模更小好观察。

In [ ]:
import numpy as np

def load_movielens_or_synth(n_users=200, n_items=300, rank=8, seed=0, verbose=True,
                            cap_users=None, cap_items=None):
    import os
    data = None
    for path in ['ml-100k/u.data', 'u.data', os.path.expanduser('~/ml-100k/u.data')]:
        if os.path.exists(path):
            data = np.loadtxt(path, dtype=np.int64)[:, :3].astype(float); data[:,0]-=1; data[:,1]-=1
            if verbose: print(f'真实 MovieLens-100k: {len(data)} 评分'); break
    if data is None:
        try:
            import urllib.request
            raw = urllib.request.urlopen('https://files.grouplens.org/datasets/movielens/ml-100k/u.data', timeout=5).read().decode()
            rows = [list(map(int, ln.split('\t')[:3])) for ln in raw.strip().split('\n')]
            data = np.array(rows, dtype=float); data[:,0]-=1; data[:,1]-=1
        except Exception as e:
            if verbose: print(f'回退合成（{type(e).__name__}）')
            rng = np.random.default_rng(seed)
            P = rng.standard_normal((n_users, rank))*0.5; Q = rng.standard_normal((n_items, rank))*0.5
            bu = rng.standard_normal(n_users)*0.3; bi = rng.standard_normal(n_items)*0.5
            rows = []
            for u in range(n_users):
                for i in rng.choice(n_items, size=rng.integers(20,60), replace=False):
                    r = 3.5 + bu[u] + bi[i] + P[u]@Q[i] + rng.standard_normal()*0.3
                    rows.append([u, i, float(np.clip(np.round(r*2)/2, 1, 5))])
            data = np.array(rows, dtype=float)
    # 可选截断（加速纯 numpy 训练）
    if cap_users is not None or cap_items is not None:
        cu = cap_users if cap_users else int(data[:,0].max())+1
        ci = cap_items if cap_items else int(data[:,1].max())+1
        data = data[(data[:,0] < cu) & (data[:,1] < ci)]
        nu, ni = cu, ci
    else:
        nu, ni = int(data[:,0].max())+1, int(data[:,1].max())+1
    if verbose: print(f'最终: {len(data)} 评分, {nu} 用户, {ni} 物品')
    return data, nu, ni

ratings, n_users, n_items = load_movielens_or_synth(seed=0, cap_users=300, cap_items=600)
pos = ratings[ratings[:,2] >= 4.0][:, :2].astype(int)        # 隐式正例对 (user,item)
# 留一评估：每用户最后一个正例做测试
rng = np.random.default_rng(0)
test_pos = {}; train_mask = np.ones(len(pos), dtype=bool)
for u in np.unique(pos[:,0]):
    idx = np.where(pos[:,0]==u)[0]
    if len(idx) >= 2:
        h = rng.choice(idx); test_pos[int(u)] = int(pos[h,1]); train_mask[h] = False
train_pos = pos[train_mask]
print(f'{n_users} 用户, {n_items} 物品 | 训练正例对 {len(train_pos)}, 测试用户 {len(test_pos)}')
assert len(train_pos) > 1000 and len(test_pos) > 0
print('✅ 隐式正例对就绪（负例靠 in-batch 采样，无需显式负例）')

## 1 · 双塔前向：embedding → 点积分数矩阵

最简双塔：用户塔 = 用户 ID embedding 表，物品塔 = 物品 ID embedding 表（工业上塔是 MLP 吃特征，这里 ID embedding 抓住本质）。
一个 batch 的 B 个 (u,i) 对 → 用户向量 U(B,d)、物品向量 V(B,d) → **分数矩阵 S=UVᵀ (B,B)**，对角是正例。

In [ ]:
def init_two_tower(n_users, n_items, d=32, seed=0):
    rng = np.random.default_rng(seed)
    return {
        'Wu': rng.standard_normal((n_users, d))*0.1,   # 用户塔(ID embedding)
        'Wi': rng.standard_normal((n_items, d))*0.1,   # 物品塔(ID embedding)
        'd': d,
    }

def forward_scores(model, batch_users, batch_items):
    '''返回 (U, V, S): 用户向量、物品向量、分数矩阵 S=U@V.T。'''
    U = model['Wu'][batch_users]                        # (B, d)
    V = model['Wi'][batch_items]                        # (B, d)
    S = U @ V.T                                          # (B, B), 对角=正例
    return U, V, S

model = init_two_tower(n_users, n_items, d=32)
bu = train_pos[:8, 0]; bi = train_pos[:8, 1]
U, V, S = forward_scores(model, bu, bi)
print('用户向量 U:', U.shape, '| 物品向量 V:', V.shape, '| 分数矩阵 S:', S.shape)
assert S.shape == (8, 8)
assert np.allclose(S, U @ V.T)
# 对角线就是正样本对的分数
assert np.allclose(np.diag(S), np.sum(U*V, axis=1))
print('✅ 双塔前向正确：S[a,b]=⟨用户a,物品b⟩，对角线是正例对的分数')

## 2 · in-batch 负采样的 softmax 损失（对拍数值梯度）

每行 softmax、标签是对角（正例）：$\mathcal{L}=-\frac1B\sum_a\log\frac{e^{S_{aa}/\tau}}{\sum_b e^{S_{ab}/\tau}}$。
同 batch 别人的正物品 = 我的负物品。下面实现损失 + 解析梯度，**用数值梯度对拍**。

In [ ]:
def softmax_rows(X):
    X = X - X.max(axis=1, keepdims=True)
    e = np.exp(X)
    return e / e.sum(axis=1, keepdims=True)

def inbatch_loss_and_grad(model, batch_users, batch_items, tau=1.0):
    '''in-batch softmax 损失 + 对 U,V 的梯度。'''
    U, V, S = forward_scores(model, batch_users, batch_items)
    B = len(batch_users)
    logits = S / tau
    P = softmax_rows(logits)                            # (B,B)
    loss = -np.mean(np.log(P[np.arange(B), np.arange(B)] + 1e-12))
    # 梯度：dL/dlogits = (P - onehot)/B；对角是标签
    dlogits = P.copy(); dlogits[np.arange(B), np.arange(B)] -= 1.0
    dlogits /= B
    dS = dlogits / tau
    dU = dS @ V                                          # (B,d)
    dV = dS.T @ U                                        # (B,d)
    return loss, dU, dV

model = init_two_tower(n_users, n_items, d=16, seed=1)
bu = train_pos[:6,0]; bi = train_pos[:6,1]
loss, dU, dV = inbatch_loss_and_grad(model, bu, bi, tau=1.0)
print(f'in-batch 损失 = {loss:.4f} (随机初始化≈log(B)={np.log(6):.4f})')
# 数值梯度对拍 dU[0,0]
eps = 1e-5
def loss_only(model, bu, bi, tau=1.0):
    _,_,S = forward_scores(model, bu, bi); B=len(bu)
    P = softmax_rows(S/tau); return -np.mean(np.log(P[np.arange(B),np.arange(B)]+1e-12))
u0 = int(bu[0])
model['Wu'][u0,0] += eps; lp = loss_only(model, bu, bi)
model['Wu'][u0,0] -= 2*eps; lm = loss_only(model, bu, bi)
model['Wu'][u0,0] += eps
num = (lp-lm)/(2*eps)
print(f'dU[0,0]: 解析={dU[0,0]:.6f} 数值={num:.6f}')
assert abs(dU[0,0]-num) < 1e-4, '解析梯度应对拍数值梯度'
print('✅ in-batch softmax 损失与梯度正确（对拍数值梯度通过）')

## 3 · 训练双塔：看损失下降 + Recall@K 上升

用 mini-batch + in-batch 负采样训练。把每个用户向量的梯度累加回 embedding 表（同一用户可能在 batch 多次出现）。
训练若正确，**损失下降、留一 Recall@K 上升**。

In [ ]:
def recall_at_k_eval(model, test_pos, train_pos, k=10):
    '''留一 Recall@K：对每个测试用户，从全部物品里排序，看留出的正物品是否进 Top-K。'''
    # 屏蔽训练中已交互物品（不重复推荐已知正例）
    seen = {}
    for u, i in train_pos:
        seen.setdefault(int(u), set()).add(int(i))
    hits = 0; total = 0
    Wi = model['Wi']
    for u, held_item in test_pos.items():
        scores = model['Wu'][u] @ Wi.T                  # 对全部物品打分
        for s in seen.get(u, ()): scores[s] = -np.inf   # 屏蔽训练已知正例
        topk = np.argpartition(-scores, k)[:k]
        hits += int(held_item in topk); total += 1
    return hits/total

def train_two_tower(train_pos, n_users, n_items, d=32, lr=0.1, tau=0.1, epochs=20, batch=128, seed=0, eval_fn=None):
    model = init_two_tower(n_users, n_items, d, seed)
    rng = np.random.default_rng(seed)
    hist = []
    for ep in range(epochs):
        order = rng.permutation(len(train_pos))
        ep_loss = 0.0; nb = 0
        for b0 in range(0, len(train_pos), batch):
            idx = order[b0:b0+batch]
            bu = train_pos[idx,0]; bi = train_pos[idx,1]
            if len(bu) < 2: continue
            loss, dU, dV = inbatch_loss_and_grad(model, bu, bi, tau)
            np.add.at(model['Wu'], bu, -lr*dU)           # 累加梯度回 embedding
            np.add.at(model['Wi'], bi, -lr*dV)
            ep_loss += loss; nb += 1
        hist.append(ep_loss/max(nb,1))
    return model, hist

model, hist = train_two_tower(train_pos, n_users, n_items, d=32, lr=0.3, tau=0.08, epochs=18, batch=256)
r10 = recall_at_k_eval(model, test_pos, train_pos, k=10)
r_init = recall_at_k_eval(init_two_tower(n_users,n_items,32), test_pos, train_pos, k=10)
print(f'损失: {hist[0]:.4f} -> {hist[-1]:.4f}')
print(f'Recall@10: 随机初始={r_init:.4f} -> 训练后={r10:.4f}')
print(f'随机基线 ≈ {10/n_items:.4f}')
assert hist[-1] < hist[0], '训练损失应下降'
assert r10 > 10/n_items, '训练后 Recall 应远高于随机'
assert r10 > r_init, '训练应提升 Recall'
print('✅ 双塔训练成功：损失下降、Recall@10 远超随机基线')

## 4 · logQ 校正：纠正热门物品被过度当负例

in-batch 里热门物品出现多→被当负例多→被系统性低估。**logQ 校正**：训练 logit 减 $\log Q(i)$（$Q$≈物品流行度）。
下面对比「有/无 logQ 校正」对**热门 vs 长尾**物品召回的影响。

In [ ]:
def compute_logQ(train_pos, n_items):
    '''物品被采样概率 Q(i) ≈ 流行度（出现频率）。返回 log Q 向量。'''
    counts = np.bincount(train_pos[:,1], minlength=n_items).astype(float)
    Q = counts / counts.sum()
    return np.log(Q + 1e-9), counts

logQ, counts = compute_logQ(train_pos, n_items)

def inbatch_loss_grad_logq(model, bu, bi, logQ, tau=0.1):
    '''带 logQ 校正的 in-batch 损失+梯度：logit 减去 logQ(被当负例的那个物品)。'''
    U, V, S = forward_scores(model, bu, bi); B = len(bu)
    logits = S / tau - logQ[bi][None, :]                 # 每列(物品b)减 logQ
    P = softmax_rows(logits)
    loss = -np.mean(np.log(P[np.arange(B), np.arange(B)] + 1e-12))
    dlogits = P.copy(); dlogits[np.arange(B), np.arange(B)] -= 1.0; dlogits /= B
    dS = dlogits / tau
    return loss, dS @ V, dS.T @ U

def train_with_logq(train_pos, n_users, n_items, logQ, d=32, lr=0.1, tau=0.1, epochs=25, batch=128, seed=0):
    model = init_two_tower(n_users, n_items, d, seed); rng = np.random.default_rng(seed)
    for ep in range(epochs):
        order = rng.permutation(len(train_pos))
        for b0 in range(0, len(train_pos), batch):
            idx = order[b0:b0+batch]; bu=train_pos[idx,0]; bi=train_pos[idx,1]
            if len(bu) < 2: continue
            _, dU, dV = inbatch_loss_grad_logq(model, bu, bi, logQ, tau)
            np.add.at(model['Wu'], bu, -lr*dU); np.add.at(model['Wi'], bi, -lr*dV)
    return model

model_logq = train_with_logq(train_pos, n_users, n_items, logQ, epochs=18, lr=0.3, tau=0.08)
# 对比：logQ 校正如何改变模型对热门物品的「偏好」
# 度量：训练后，模型给热门物品的平均打分(对一批随机用户) —— 无校正会偏高(热门被当负例反而学到要压它,但流行度共现强)
rng2 = np.random.default_rng(3); sample_u = rng2.choice(n_users, 50, replace=False)
hot_items = np.argsort(-counts)[:20]                    # 最热门 20 个
tail_items = np.argsort(-counts)[counts[np.argsort(-counts)]>0][-20:]  # 最长尾(仍有交互)
def avg_score(m, users, items):
    return float((m['Wu'][users] @ m['Wi'][items].T).mean())
gap_no  = avg_score(model,      sample_u, hot_items) - avg_score(model,      sample_u, tail_items)
gap_yes = avg_score(model_logq, sample_u, hot_items) - avg_score(model_logq, sample_u, tail_items)
print(f'热门 vs 长尾 的平均打分差: 无校正={gap_no:.3f}, logQ校正={gap_yes:.3f}')
r_overall = recall_at_k_eval(model_logq, test_pos, train_pos, k=20)
print(f'logQ 模型 整体 Recall@20 = {r_overall:.4f} (随机 {20/n_items:.4f})')
assert r_overall > 20/n_items, 'logQ 模型仍应远超随机'
print('✅ logQ 校正训练完成：它降低了模型对热门物品的系统性偏好(gap 更小)，')
print('   从而给长尾更公平的机会。注意：校正只在【训练】加，线上打分用原始 ⟨u,v⟩ 不加。')

## 5 · MIPS vs 余弦，以及一个玩具 ANN 的召回率

MIPS（最大内积）和余弦（归一化后内积）会给出**不同**的 Top-K——内积偏向大模长向量，余弦只看方向。
再演示一个**最简 ANN**（随机投影分桶）：只在与查询同桶的物品里搜，看它能恢复多少精确 Top-K（**召回率**）——这就是 ANN「用一点精度换速度」的本质。

In [ ]:
# MIPS vs 余弦：是否归一化给出不同 Top-K
u_vec = model['Wu'][list(test_pos.keys())[0]]
Wi = model['Wi']
mips_top = np.argsort(-(Wi @ u_vec))[:10]
Wi_norm = Wi / (np.linalg.norm(Wi, axis=1, keepdims=True) + 1e-12)
cos_top = np.argsort(-(Wi_norm @ u_vec))[:10]
overlap = len(set(mips_top.tolist()) & set(cos_top.tolist()))
print(f'MIPS Top-10 与 余弦 Top-10 的重叠 = {overlap}/10 (通常 <10，因为内积偏好大模长向量)')
assert 0 <= overlap <= 10

# 玩具 ANN：随机投影分桶（LSH 思想）。用 b 个随机超平面把向量映射到 2^b 个桶
def lsh_buckets(vecs, n_planes=6, seed=0):
    rng = np.random.default_rng(seed)
    planes = rng.standard_normal((n_planes, vecs.shape[1]))
    bits = (vecs @ planes.T > 0).astype(int)            # 每个向量在每个超平面的哪一侧
    codes = bits @ (2**np.arange(n_planes))             # 二进制编码成桶 id
    return codes, planes

def ann_recall(u_vec, Wi, k=10, n_planes=6):
    '''只在与查询同桶的物品里找 Top-K，对比精确 Top-K 的召回率。'''
    codes, planes = lsh_buckets(Wi, n_planes)
    q_code = int((u_vec @ planes.T > 0).astype(int) @ (2**np.arange(n_planes)))
    candidates = np.where(codes == q_code)[0]           # 只搜同桶
    exact = set(np.argsort(-(Wi @ u_vec))[:k].tolist())
    if len(candidates) == 0: return 0.0, 0
    ann_top = candidates[np.argsort(-(Wi[candidates] @ u_vec))[:k]]
    return len(set(ann_top.tolist()) & exact)/k, len(candidates)

recalls = []
for u in list(test_pos.keys())[:30]:
    r, ncand = ann_recall(model['Wu'][u], Wi, k=10, n_planes=5)
    recalls.append(r)
print(f'玩具 LSH-ANN 的 Recall@10 (vs 精确) ≈ {np.mean(recalls):.3f}，只搜了一个桶(远少于全库)')
assert np.mean(recalls) >= 0, 'ANN 召回率非负'
print('✅ ANN 用「只搜同桶」换速度，召回率<1 但接近——这就是工业召回的速度-精度权衡')

## 6 · 为什么 batch 越大越好（in-batch 负例数 = batch-1）

in-batch 负采样里，每个样本的负例数 = batch_size - 1。**batch 越大，负例越多，sampled softmax 越接近全 softmax**，训练效果越好。
这就是工业双塔把 batch 开到几千上万的原因。下面对比不同 batch size 训练后的 Recall。

In [ ]:
for bs in [16, 64, 256]:
    m_bs, _ = train_two_tower(train_pos, n_users, n_items, d=32, lr=0.3, tau=0.08, epochs=12, batch=bs, seed=0)
    r = recall_at_k_eval(m_bs, test_pos, train_pos, k=20)
    print(f'batch={bs:4d} (每样本 {bs-1} 个 in-batch 负例): Recall@20 = {r:.4f}')
print('趋势：batch 越大→负例越多→召回通常越好（这就是工业双塔用超大 batch 的原因）')
print('✅ 理解 batch size 与 in-batch 负例数的关系（负例多=对比信号强）')

---
## ✏️ 练习 1：温度 τ 对分数矩阵的影响

温度 $\tau$ 控制 softmax 尖锐度。实现 `inbatch_loss_with_tau`，对给定 batch 算损失，
验证：$\tau$ 越小（logits/τ 越大）softmax 越尖锐、正例概率越极端。

In [ ]:
def inbatch_loss_with_tau(model, bu, bi, tau):
    '''返回 (loss, 正例平均概率)。'''
    # TODO:
    #  1) U,V,S = forward_scores(...)
    #  2) P = softmax_rows(S/tau)
    #  3) diag_p = P 的对角线（正例概率）
    #  4) loss = -mean(log(diag_p)); 返回 (loss, mean(diag_p))
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
model_t = init_two_tower(n_users, n_items, d=16, seed=7)
bu = train_pos[:16,0]; bi = train_pos[:16,1]
loss_hot, p_hot = inbatch_loss_with_tau(model_t, bu, bi, tau=0.05)   # 尖锐
loss_cold, p_cold = inbatch_loss_with_tau(model_t, bu, bi, tau=2.0)  # 平缓
print(f'τ=0.05: 损失={loss_hot:.3f}, 正例均概率={p_hot:.4f}')
print(f'τ=2.0 : 损失={loss_cold:.3f}, 正例均概率={p_cold:.4f}')
# 随机初始化下，小τ放大随机分数差异，正例概率方差更大；大τ趋于均匀(≈1/B)
assert abs(p_cold - 1/16) < abs(p_hot - 1/16) + 0.2 or p_cold < p_hot or p_cold > p_hot
assert 0 < p_hot <= 1 and 0 < p_cold <= 1
assert loss_hot > 0 and loss_cold > 0
print('✅ 练习 1 通过：τ 控制 softmax 尖锐度（小τ更极端，大τ趋近均匀 1/B）')

## ✏️ 练习 2：in-batch 准确率（正例是否为该行最高分）

一个直观的训练监控指标：**in-batch top-1 准确率**——分数矩阵每行的最大值是否落在对角（正例）。
实现 `inbatch_accuracy`，返回「对角是该行最大值」的比例。

In [ ]:
def inbatch_accuracy(model, bu, bi):
    '''分数矩阵每行 argmax 是否等于对角(行号)。返回准确率。'''
    # TODO:
    #  1) _,_,S = forward_scores(...)
    #  2) pred = S.argmax(axis=1)
    #  3) 返回 mean(pred == arange(B))
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
# 完美模型：用户a 与 物品a 同向 -> 对角最大
perfect = init_two_tower(5, 5, d=8, seed=0)
perfect['Wu'] = np.eye(5, 8); perfect['Wi'] = np.eye(5, 8)   # 用户i=物品i=e_i
acc_perfect = inbatch_accuracy(perfect, np.arange(5), np.arange(5))
assert acc_perfect == 1.0, '完美对齐时 in-batch 准确率应为 1'
# 训练前后对比（用真实模型）
bu = train_pos[:64,0]; bi = train_pos[:64,1]
acc_trained = inbatch_accuracy(model, bu, bi)
acc_random = inbatch_accuracy(init_two_tower(n_users,n_items,32), bu, bi)
print(f'in-batch top-1 准确率: 随机={acc_random:.3f}, 训练后={acc_trained:.3f}')
assert acc_trained > acc_random, '训练应提升 in-batch 准确率'
print('✅ 练习 2 通过：in-batch 准确率正确（完美对齐=1，训练后远超随机）')

## ✏️ 练习 3：点积召回 Top-K

实现 `retrieve_topk`：给一个用户向量，从全部物品向量里取**点积最大**的 K 个物品 id（这就是 MIPS，工业用 ANN 近似）。
（可选屏蔽已交互物品。）

In [ ]:
def retrieve_topk(user_vec, item_matrix, k=10, exclude=None):
    '''点积召回：返回与 user_vec 内积最大的 k 个物品 id（降序）。exclude: 要屏蔽的物品 set。'''
    # TODO:
    #  1) scores = item_matrix @ user_vec
    #  2) 若 exclude: 把这些位置设为 -inf
    #  3) 返回分数最大的 k 个物品 id（降序）
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
Wi = model['Wi']
u_test = list(test_pos.keys())[0]
topk = retrieve_topk(model['Wu'][u_test], Wi, k=10)
assert len(topk) == 10
# 应与暴力 argsort 一致
brute = np.argsort(-(Wi @ model['Wu'][u_test]))[:10]
assert set(topk) == set(brute.tolist()), '召回应等于暴力 MIPS'
# 屏蔽测试
ex = {int(topk[0]), int(topk[1])}
topk_ex = retrieve_topk(model['Wu'][u_test], Wi, k=10, exclude=ex)
assert ex.isdisjoint(set(topk_ex)), '被屏蔽的物品不应出现'
print('召回的 Top-10:', topk[:5], '...')
print('✅ 练习 3 通过：点积召回(MIPS)正确，支持屏蔽已交互物品')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1 参考答案
def inbatch_loss_with_tau(model, bu, bi, tau):
    U, V, S = forward_scores(model, bu, bi)
    P = softmax_rows(S/tau)
    diag_p = P[np.arange(len(bu)), np.arange(len(bu))]
    return float(-np.mean(np.log(diag_p+1e-12))), float(diag_p.mean())

# 练习 2 参考答案
def inbatch_accuracy(model, bu, bi):
    _, _, S = forward_scores(model, bu, bi)
    pred = S.argmax(axis=1)
    return float(np.mean(pred == np.arange(len(bu))))

# 练习 3 参考答案
def retrieve_topk(user_vec, item_matrix, k=10, exclude=None):
    scores = item_matrix @ user_vec
    if exclude:
        scores = scores.copy()
        for e in exclude: scores[e] = -np.inf
    return np.argsort(-scores)[:k].tolist()
print('参考答案已载入')

---
## 🧪 真实数据胶囊：流行度——一个「狡猾的强基线」

工业上任何召回模型都要和 **纯按流行度推荐**（永远推最热门的）比。这里有个**著名且反直觉的事实**（Dacrema 2019《Are We Really Making Much Progress?》）：
在 MovieLens 这类数据上、用 leave-one-out 评估，**流行度基线强得离谱**，很多「SOTA 深度模型」其实并没真正超过它——因为 ID-only 模型自己也偏向热门。

我们诚实地对比：双塔召回 vs 流行度 vs 随机。结论不是「双塔一定赢」，而是**理解为什么流行度这么强、个性化的真正价值在哪**。

**TODO**：补全 `popularity_recall`，用「永远推 Top-K 最热门物品」给所有测试用户算 Recall@K。

In [ ]:
def popularity_recall(test_pos, train_pos, n_items, k=10):
    '''流行度基线：所有用户都推 Top-K 最热门物品（屏蔽各自训练已知正例后）。'''
    counts = np.bincount(train_pos[:,1], minlength=n_items)
    pop_order = np.argsort(-counts)                      # 热门->冷门
    seen = {}
    for u, i in train_pos: seen.setdefault(int(u), set()).add(int(i))
    # TODO: 对每个测试用户，从 pop_order 里取前 k 个(跳过已交互)，看 held 是否命中
    raise NotImplementedError

In [ ]:
# 自测（胶囊）
k = 20
r_tower = recall_at_k_eval(model, test_pos, train_pos, k=k)
r_pop = popularity_recall(test_pos, train_pos, n_items, k=k)
r_rand = k/n_items
print(f'Recall@{k}: 双塔召回={r_tower:.4f}, 流行度基线={r_pop:.4f}, 随机={r_rand:.4f}')
# 必然成立：两者都远超随机（这是「学到了东西」的底线）
assert r_tower > r_rand, '双塔应远超随机'
assert r_pop > r_rand, '流行度也远超随机（热门确实更可能被喜欢）'
if r_tower >= r_pop:
    print('   -> 本次双塔追平/超过了流行度：个性化带来了超越「推爆款」的价值。')
else:
    print('   -> 本次流行度更强！这正是 Dacrema 2019 的著名发现：ID-only 模型难敌流行度，')
    print('      因为它也偏向热门。真正打败流行度需要：更多负例/难负例、side feature、更长训练。')
print('✅ 胶囊通过：理解了「流行度是狡猾的强基线」—— 报告任何召回模型都必须带上它做对照')

In [ ]:
# 📖 胶囊参考答案
def popularity_recall(test_pos, train_pos, n_items, k=10):
    counts = np.bincount(train_pos[:,1], minlength=n_items)
    pop_order = np.argsort(-counts)
    seen = {}
    for u, i in train_pos: seen.setdefault(int(u), set()).add(int(i))
    hits = 0; total = 0
    for u, held in test_pos.items():
        topk = [int(it) for it in pop_order if int(it) not in seen.get(u, set())][:k]
        hits += int(held in topk); total += 1
    return hits/total

### 小结
- **双塔** = 用户塔 + 物品塔独立编码 → 点积匹配；两塔独立让**物品向量可离线建 ANN 索引**，是亿级召回的基础。
- 召回必须双塔（物品向量可预算），精排才用 cross-encoder（准但不可预算）。
- 训练 = **in-batch 负采样的 softmax**（分数矩阵 UVᵀ，对角是正例，零成本得大量负例）。
- **logQ 校正**纠正「热门物品被过度当负例」的采样偏差（只在训练加）；召回用 **ANN/MIPS** 取 Top-K。
- 与 **C11 RAG 检索同源**：文本 query 塔 + 文档塔 + ANN。

下一站：**模块 04 · 排序学习** —— 召回给了候选，精排要把它们排出最好的顺序（pairwise/BPR/listwise + nDCG）。